# Séance 10 — API REST avec FastAPI

## 🛟 Notebook « point de reprise »

**À quoi sert ce notebook ?** Si tu as manqué la séance précédente, si ton code
ne marche pas, ou si tu t'es perdu·e en route : **ouvre celui-ci**. Le code de
départ est déjà écrit et fonctionne. Tu n'as jamais besoin d'avoir réussi
l'exercice d'avant pour suivre celui d'aujourd'hui.

**Comment l'utiliser ?**
1. Exécute les cellules du haut sans les modifier (elles remettent tout en place).
2. Descends jusqu'aux cellules `# ✏️ À TOI DE JOUER`.
3. Écris ton code à la place des `...`.

**Raccourci** : `Maj + Entrée` exécute une cellule.

---


> ⚠️ **Cette séance se déroule dans VS Code.** Une API se lance depuis un
> terminal (`uvicorn main:app --reload`), pas depuis un notebook.
> Ce notebook sert de référence et de rattrapage ; le projet complet est
> dans `fil-rouge/v7-api/`.
>
> ⚠️ **FastAPI a supprimé le support de Pydantic v1.** Tout tutoriel montrant
> `@validator`, `.dict()` ou `class Config` est périmé.


## 1. REST : vous l'avez déjà écrit en séance 5

| Verbe | Adresse | Sens | Équivalent séance 5 |
|---|---|---|---|
| GET | `/opportunites` | lister | `carnet.toutes()` |
| GET | `/opportunites/12` | consulter | `carnet[12]` |
| POST | `/opportunites` | créer | `carnet.ajouter(...)` |
| DELETE | `/opportunites/12` | supprimer | `carnet.supprimer(12)` |

**L'API n'ajoute pas de logique. Elle ajoute une porte d'entrée.**


## 2. Pydantic v2 : le videur à l'entrée

In [ ]:
from datetime import date

from pydantic import BaseModel, Field, computed_field, field_validator


class Opportunite(BaseModel):
    titre: str = Field(min_length=3, max_length=200)
    pays: str = Field(min_length=2)
    deadline: date

    @field_validator("pays")          # ⚠️ v2 : field_validator, PAS validator
    @classmethod
    def normaliser_pays(cls, valeur: str) -> str:
        return valeur.strip().title()

    @computed_field
    @property
    def jours_restants(self) -> int:
        return (self.deadline - date.today()).days


o = Opportunite(titre="Bourse Smarts-Up", pays="  france ", deadline="2027-01-15")
print(o.model_dump())      # ⚠️ v2 : model_dump(), PAS .dict()


In [ ]:
# La validation refuse AVANT que ton code ne s'exécute :
from pydantic import ValidationError

try:
    Opportunite(titre="ab", pays="Maroc", deadline="2027-01-15")
except ValidationError as erreur:
    print(erreur)


## 3. ⚠️ Pydantic v1 → v2 : la table à garder sous les yeux

| ❌ v1 (obsolète) | ✅ v2 |
|---|---|
| `@validator("champ")` | `@field_validator("champ")` |
| `model.dict()` | `model.model_dump()` |
| `model.json()` | `model.model_dump_json()` |
| `Model.parse_obj(d)` | `Model.model_validate(d)` |
| `class Config:` | `model_config = ConfigDict(...)` |
| `Optional[str] = None` | `str | None = None` |


---
# ✏️ À TOI DE JOUER — L'API OpportuniTrack

Dans VS Code :
```bash
uv add fastapi uvicorn
uv run uvicorn main:app --reload
```
puis ouvre **http://127.0.0.1:8000/docs** et clique sur « Try it out ».

Routes à écrire : `GET /opportunites`, `GET /opportunites/{id}`,
`POST /opportunites`, `DELETE /opportunites/{id}`, `GET /statistiques`.

Corrigé complet : `fil-rouge/v7-api/`.


In [ ]:
# ✏️ Squelette de départ (à copier dans main.py)
from fastapi import Depends, FastAPI, HTTPException, status

app = FastAPI(title="OpportuniTrack API", version="1.0.0")


@app.get("/opportunites")
def lister(pays: str | None = None):
    """Cette docstring devient la description de la route dans /docs."""
    ...


@app.get("/opportunites/{opp_id}")
def consulter(opp_id: int):
    ...
    # raise HTTPException(status.HTTP_404_NOT_FOUND, "Introuvable")
